<a href="https://colab.research.google.com/github/Danieladunaev/Danieladunaev/blob/main/segmentation_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentation Validation Agent

**Purpose:** assess whether four client segments are sufficiently distinct, statistically and practically meaningful, stable over time, and associated with materially different default behaviour.



## 1. Methodology and Key Questions

This notebook evaluates whether the 4 customer segments are valid and statistically sound.

### Core Questions:
1. Are segment labels valid and large enough for reliable analysis?
2. Which client characteristics (e.g., income, age, credit history) separate the segments?
3. Are the differences real and consistent, or just random noise (controlling for false discoveries)?
4.  Do default rates differ meaningfully across segments?
5.  Does knowing a customer's segment tell us anything new about their risk, beyond standard attributes like age and income?
6.  Does segment behavior remain consistent across different loan underwriting years?

---

### The 4 Synthetic Customer Segments:
- **Segment 1 — Established / Low Risk:** Older clients with longer employment, strong credit history, and high credit scores.
- **Segment 2 — Young / Thin File:** Younger clients with limited credit history and smaller loan amounts.
- **Segment 3 — Leveraged / High Risk:** High debt-to-income ratios, high credit utilization, and past late payments.
- **Segment 4 — Affluent / High Exposure:** High earners with large loans and moderate risk indicators.

*Note: These definitions are initial working hypotheses. The analysis below tests whether the actual data supports them.*

In [ ]:
# Colab setup: uncomment only if a package is missing.
# %pip install -q pandas numpy scipy statsmodels scikit-learn seaborn matplotlib

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", palette="colorblind")

SEGMENT_COL = "Segment"
TARGET_COL = "Default"
ID_COL = "Client_ID"
YEAR_COL = "Loan_Underwriting_Year"
ALPHA = 0.05
PRACTICAL_SMD = 0.20
PRACTICAL_DEFAULT_RATE_GAP_PP = 2.0


## 2. Load data

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print(f"Loaded: {file_name}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

df.head()

### What we can see and learn from this output

- The updated file loaded correctly: **30,000 rows and 15 columns**.
- Each row represents one client. The columns contain the client's characteristics, underwriting year, segment, and default result.
- The first five rows are only a quick preview. They show that clients from different segments and with different default results are present.
- Colab renamed the uploaded file with `(2)` because a file with the same name was already in the session. This does not change the data.

**Main lesson:** the correct 30,000-client dataset is ready for analysis.

## 3. Data-quality and segmentation-assumption gates

In [ ]:
required = {ID_COL, SEGMENT_COL, TARGET_COL, YEAR_COL}
missing_required = sorted(required - set(df.columns))
duplicate_ids = int(df[ID_COL].duplicated().sum()) if ID_COL in df else None
null_cells = int(df.isna().sum().sum())
observed_segments = sorted(df[SEGMENT_COL].dropna().unique().tolist()) if SEGMENT_COL in df else []
target_values = sorted(df[TARGET_COL].dropna().unique().tolist()) if TARGET_COL in df else []

validation = pd.DataFrame({
    "Check": ["Required columns", "No missing cells", "Unique client IDs", "Segments are exactly 1-4", "Default is binary 0/1"],
    "Result": [
        not missing_required,
        null_cells == 0,
        duplicate_ids == 0,
        observed_segments == [1, 2, 3, 4],
        target_values == [0, 1],
    ],
    "Detail": [
        f"Missing: {missing_required}" if missing_required else "All present",
        f"Missing cells: {null_cells:,}",
        f"Duplicate IDs: {duplicate_ids:,}",
        f"Observed: {observed_segments}",
        f"Observed: {target_values}",
    ],
})
display(validation)
assert validation["Result"].all(), "Data-quality gate failed; review the table above."


### What we can see and learn from this output

- Every check returned `True`.
- All required columns are present.
- There are **0 missing cells** and **0 duplicate client IDs**.
- The segment column contains exactly **1, 2, 3, and 4**.
- The default column contains only **0 and 1**.

**Main lesson:** the data passes the basic quality checks, so we can continue without first cleaning missing values or duplicate clients.

## 4. Dataset structure and segment reliability

In [ ]:
display(df.info())

# A segment is useful only if its outcome estimate is reliable.
# These thresholds are tied to the analysis, rather than to an arbitrary total size.
MIN_DEFAULTS = 20
MIN_NON_DEFAULTS = 20
MAX_CI_WIDTH_PP = 4.0  # no more than +/- 2 percentage points at 95% confidence

segment_quality = (
    df.groupby(SEGMENT_COL)[TARGET_COL]
    .agg(Default_Count="sum", Client_Count="count", Default_Rate="mean")
)
segment_quality["Non_Default_Count"] = (
    segment_quality["Client_Count"] - segment_quality["Default_Count"]
)
segment_quality["Share_%"] = 100 * segment_quality["Client_Count"] / len(df)

intervals = segment_quality.apply(
    lambda row: proportion_confint(
        row["Default_Count"], row["Client_Count"], alpha=ALPHA, method="wilson"
    ),
    axis=1,
)
segment_quality["Default_Rate_CI_Lower"] = [interval[0] for interval in intervals]
segment_quality["Default_Rate_CI_Upper"] = [interval[1] for interval in intervals]
segment_quality["CI_Width_pp"] = 100 * (
    segment_quality["Default_Rate_CI_Upper"]
    - segment_quality["Default_Rate_CI_Lower"]
)

segment_quality["Low_Default_Count_Warning"] = (
    segment_quality["Default_Count"] < MIN_DEFAULTS
)
segment_quality["Low_Non_Default_Count_Warning"] = (
    segment_quality["Non_Default_Count"] < MIN_NON_DEFAULTS
)
segment_quality["Imprecise_Rate_Warning"] = (
    segment_quality["CI_Width_pp"] > MAX_CI_WIDTH_PP
)
segment_quality["Segment_Adequacy_Warning"] = segment_quality[
    [
        "Low_Default_Count_Warning",
        "Low_Non_Default_Count_Warning",
        "Imprecise_Rate_Warning",
    ]
].any(axis=1)

# Keep this name for the export cell later in the notebook.
segment_sizes = segment_quality

display(
    segment_quality.style.format(
        {
            "Default_Rate": "{:.2%}",
            "Share_%": "{:.2f}%",
            "Default_Rate_CI_Lower": "{:.2%}",
            "Default_Rate_CI_Upper": "{:.2%}",
            "CI_Width_pp": "{:.2f}",
        }
    )
)

ax = sns.barplot(
    x=segment_quality.index.astype(str),
    y=segment_quality["Client_Count"],
    color="#4472C4",
)
ax.set(title="Client population by segment", xlabel="Segment", ylabel="Clients")
for patch, (_, row) in zip(ax.patches, segment_quality.iterrows()):
    ax.annotate(
        f"{int(row['Client_Count']):,}\n({row['Share_%']:.1f}%)",
        (patch.get_x() + patch.get_width() / 2, patch.get_height()),
        ha="center",
        va="bottom",
    )
plt.show()


### What we can see and learn from this output

This check now focuses on whether each segment's **default-rate estimate is reliable**, rather than asking whether the segment has an arbitrary 100 clients.

- Every segment has far more than **20 defaults** and **20 non-defaults**, so both outcomes are properly represented.
- The 95% confidence-interval widths are approximately **0.52 points** for Segment 1, **1.24** for Segment 2, **2.54** for Segment 3, and **2.34** for Segment 4.
- All widths are below the chosen maximum of **4 percentage points** (approximately +/- 2 points around the estimated rate).
- Therefore, `Segment_Adequacy_Warning` should be `False` for all four segments after the cell is rerun.
- Segment 4 is the smallest, but its default-rate estimate is still precise enough for this analysis.

**Main lesson:** all four segments contain enough outcome information and provide sufficiently precise default-rate estimates. This is much more meaningful than checking whether the total size is below 100.

## 5. Descriptive segment profiles

In [ ]:
excluded = {ID_COL, SEGMENT_COL, TARGET_COL, YEAR_COL}
numeric_features = [c for c in df.select_dtypes(include=np.number).columns if c not in excluded]

profile = df.groupby(SEGMENT_COL)[numeric_features].agg(["mean", "median", "std"])
display(profile.round(3))

long = df.melt(id_vars=[SEGMENT_COL], value_vars=numeric_features, var_name="Feature", value_name="Value")
g = sns.FacetGrid(long, col="Feature", col_wrap=3, sharex=False, sharey=False, height=3.2)
g.map_dataframe(sns.boxplot, x=SEGMENT_COL, y="Value", color="#76A5AF", showfliers=False)
g.set_titles("{col_name}")
g.fig.subplots_adjust(top=0.94)
g.fig.suptitle("Numeric feature distributions by segment")
plt.show()


### What we can see and learn from this output

- **Segment 1** looks like an established, lower-risk group: older clients, longer employment and credit histories, lower debt and utilization, and the highest average credit score (about **735**).
- **Segment 2** is the youngest group: average age about **29**, short employment and credit histories, the lowest income, and the smallest loans.
- **Segment 3** has the clearest high-risk profile: the highest debt-to-income ratio (about **0.51**), highest utilization (about **0.71**), most late payments, and lowest average credit score (about **604**).
- **Segment 4** is the affluent, high-exposure group: the highest average income (about **147,712**) and by far the largest average loan (about **71,560**).
- The boxplots show some overlap between groups. This is realistic: clients in different segments can still have similar individual values.

**Main lesson:** the segments describe different and understandable client profiles. Segment 3 appears riskiest, while Segment 1 appears safest.

## 6. Overall numeric differentiation tests

Kruskal–Wallis is used because it does not require normally distributed variables. Epsilon-squared measures practical separation. Benjamini–Hochberg correction controls the false-discovery rate across features.

In [ ]:
def kruskal_effect_table(data, features, segment_col=SEGMENT_COL):
    rows = []
    k = data[segment_col].nunique()
    n = len(data)
    for feature in features:
        groups = [g[feature].to_numpy() for _, g in data.groupby(segment_col)]
        h, p = stats.kruskal(*groups)
        epsilon_sq = max(0.0, (h - k + 1) / (n - k))
        rows.append({"Feature": feature, "Kruskal_H": h, "P_Value": p, "Epsilon_Squared": epsilon_sq})
    result = pd.DataFrame(rows)
    result["Adjusted_P_Value"] = multipletests(result["P_Value"], method="fdr_bh")[1]
    result["Statistically_Significant"] = result["Adjusted_P_Value"] < ALPHA
    result["Practical_Effect"] = pd.cut(
        result["Epsilon_Squared"], [-np.inf, 0.01, 0.06, 0.14, np.inf],
        labels=["Negligible", "Small", "Medium", "Large"]
    )
    return result.sort_values("Epsilon_Squared", ascending=False)

numeric_tests = kruskal_effect_table(df, numeric_features)
display(numeric_tests.style.format({"Kruskal_H": "{:.2f}", "P_Value": "{:.3g}", "Adjusted_P_Value": "{:.3g}", "Epsilon_Squared": "{:.3f}"}))

sns.barplot(data=numeric_tests, y="Feature", x="Epsilon_Squared", hue="Practical_Effect", dodge=False)
plt.axvline(0.06, color="orange", linestyle="--", label="Medium threshold")
plt.axvline(0.14, color="red", linestyle="--", label="Large threshold")
plt.title("Practical segment separation by feature")
plt.legend()
plt.show()


### What we can see and learn from this output

- All 11 variables have adjusted p-values close to zero, so their distributions differ across the segments.
- More importantly, every epsilon-squared value is above the **0.14 large-effect line**. The differences are therefore practically large, not only statistically significant because of the 30,000-row sample.
- `Income` gives the strongest separation (**0.606**).
- `Credit_Bureau_Score` (**0.521**) and `Age` (**0.495**) are also major separators.
- `Late_Payments_30d` is the weakest separator in this table (**0.314**), but it is still a large effect.

**Main lesson:** the segmentation is strongly connected to every tested characteristic. In real data, such consistently large separation would deserve review to make sure the segment rules are not too rigid or circular.

## 7. Pairwise standardized mean differences

Absolute standardized mean difference (SMD) provides a scale-independent measure. A common review threshold is 0.20 for a potentially meaningful difference.

In [ ]:
def pairwise_smd(data, features, segment_col=SEGMENT_COL):
    rows = []
    segments = sorted(data[segment_col].unique())
    for i, a in enumerate(segments):
        for b in segments[i+1:]:
            for feature in features:
                x = data.loc[data[segment_col] == a, feature]
                y = data.loc[data[segment_col] == b, feature]
                pooled = np.sqrt((x.var(ddof=1) + y.var(ddof=1)) / 2)
                smd = (x.mean() - y.mean()) / pooled if pooled else 0.0
                rows.append({"Segment_A": a, "Segment_B": b, "Feature": feature, "SMD": smd, "Abs_SMD": abs(smd)})
    return pd.DataFrame(rows)

smd_table = pairwise_smd(df, numeric_features)
smd_pivot = smd_table.assign(Pair=lambda x: x["Segment_A"].astype(str)+" vs "+x["Segment_B"].astype(str)).pivot(index="Feature", columns="Pair", values="Abs_SMD")
plt.figure(figsize=(9, max(5, len(numeric_features)*0.45)))
sns.heatmap(smd_pivot, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, center=PRACTICAL_SMD)
plt.title("Absolute standardized mean differences")
plt.show()

display(smd_table.sort_values("Abs_SMD", ascending=False).head(30).style.format({"SMD": "{:.3f}", "Abs_SMD": "{:.3f}"}))


### What we can see and learn from this output

- Each heatmap column compares one pair of segments. Darker red means a larger difference.
- The largest pairwise differences are between Segments 2 and 4 for `Loan_Amount` (**3.35**) and `Income` (**3.34**).
- Segments 1 and 3 differ strongly in utilization (**3.10**), credit score (**3.00**), and debt-to-income (**2.94**).
- Segment 1 versus Segment 2 differs strongly in age (**2.51**), employment history (**2.38**), and credit-history length (**2.37**).
- A few comparisons are weak: late payments for Segments 2 and 4 (**0.15**) and credit-history length for Segments 1 and 4 (**0.17**) are below the usual **0.20** threshold.

**Main lesson:** most segment pairs are clearly different, but not every variable separates every pair. This is why the pairwise heatmap is more informative than looking only at the overall test.

## 8. Default behaviour by segment

In [ ]:
default_summary = df.groupby(SEGMENT_COL)[TARGET_COL].agg(["sum", "count", "mean"]).rename(columns={"sum":"Defaults", "count":"Clients", "mean":"Default_Rate"})
intervals = default_summary.apply(lambda r: proportion_confint(r["Defaults"], r["Clients"], alpha=ALPHA, method="wilson"), axis=1)
default_summary["CI_Lower"] = [x[0] for x in intervals]
default_summary["CI_Upper"] = [x[1] for x in intervals]
display(default_summary.style.format({"Default_Rate":"{:.2%}", "CI_Lower":"{:.2%}", "CI_Upper":"{:.2%}"}))

plot_df = default_summary.reset_index()
plt.errorbar(plot_df[SEGMENT_COL], 100*plot_df["Default_Rate"], yerr=[100*(plot_df["Default_Rate"]-plot_df["CI_Lower"]), 100*(plot_df["CI_Upper"]-plot_df["Default_Rate"])], fmt="o", capsize=5)
plt.xticks(sorted(df[SEGMENT_COL].unique()))
plt.xlabel("Segment"); plt.ylabel("Default rate (%)"); plt.title("Default rate and 95% Wilson confidence interval")
plt.show()

contingency = pd.crosstab(df[SEGMENT_COL], df[TARGET_COL])
chi2, p_value, _, _ = stats.chi2_contingency(contingency)
n = contingency.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * min(contingency.shape[0]-1, contingency.shape[1]-1)))
print(f"Overall chi-square p-value: {p_value:.4g} | Cramer's V: {cramers_v:.3f}")


### What we can see and learn from this output

- Segment 1 has **279 defaults out of 12,600 clients**, for a default rate of **2.21%**.
- Segment 2 has **773 defaults out of 8,400**, for **9.20%**.
- Segment 3 has **3,401 defaults out of 5,700**, for **59.67%**.
- Segment 4 has **452 defaults out of 3,300**, for **13.70%**.
- The confidence intervals are narrow and do not overlap much, especially for Segment 3.
- Cramer's V is **0.577**, showing a strong relationship between segment and default.

**Main lesson:** the segments clearly identify groups with different observed risk. Segment 3 is much riskier and Segment 1 is much safer. This is an association; it does not prove that being assigned to a segment causes default.

## 9. Pairwise default-rate comparisons

Two-proportion z-tests are Holm-adjusted. Both adjusted significance and a minimum absolute gap of 2 percentage points are required for a practically meaningful flag.

In [ ]:
segments = sorted(df[SEGMENT_COL].unique())
pair_rows = []
for i, a in enumerate(segments):
    for b in segments[i+1:]:
        da = df[df[SEGMENT_COL] == a][TARGET_COL]
        db = df[df[SEGMENT_COL] == b][TARGET_COL]
        z, p = proportions_ztest([da.sum(), db.sum()], [len(da), len(db)])
        pair_rows.append({"Segment_A": a, "Segment_B": b, "Rate_A": da.mean(), "Rate_B": db.mean(), "Gap_pp": 100*(da.mean()-db.mean()), "P_Value": p})

pairwise_default = pd.DataFrame(pair_rows)
pairwise_default["Adjusted_P_Value"] = multipletests(pairwise_default["P_Value"], method="holm")[1]
pairwise_default["Material_Difference"] = (pairwise_default["Adjusted_P_Value"] < ALPHA) & (pairwise_default["Gap_pp"].abs() >= PRACTICAL_DEFAULT_RATE_GAP_PP)
display(pairwise_default.style.format({"Rate_A":"{:.2%}", "Rate_B":"{:.2%}", "Gap_pp":"{:.2f}", "P_Value":"{:.3g}", "Adjusted_P_Value":"{:.3g}"}))


### What we can see and learn from this output

- `Material_Difference` is `True` for all six segment pairs.
- The largest gap is Segment 1 versus Segment 3: **57.45 percentage points**.
- Segment 2 versus Segment 3 also has a very large gap: **50.46 points**.
- The smallest gap is Segment 2 versus Segment 4: **4.49 points**, which is still above the notebook's 2-point materiality rule.
- All Holm-adjusted p-values remain below 0.05.

**Main lesson:** every segment has meaningfully different raw default behaviour from every other segment.

## 10. Does segment add information after controlling for risk characteristics?

The adjusted logistic regression controls for all numeric characteristics and underwriting year. Segment 1 is the reference category. Odds ratios above 1 imply higher adjusted odds of default than Segment 1.

In [ ]:
model_df = df.copy()
standardized = []
for feature in numeric_features:
    name = f"z_{feature}"
    model_df[name] = (model_df[feature] - model_df[feature].mean()) / model_df[feature].std(ddof=0)
    standardized.append(name)

formula = f"{TARGET_COL} ~ C({SEGMENT_COL}, Treatment(reference=1)) + " + " + ".join(standardized) + f" + C({YEAR_COL})"
model = smf.logit(formula, data=model_df).fit(disp=False)

segment_terms = [term for term in model.params.index if term.startswith(f"C({SEGMENT_COL}")]
adjusted_segment_results = pd.DataFrame({
    "Term": segment_terms,
    "Coefficient": model.params[segment_terms],
    "Odds_Ratio": np.exp(model.params[segment_terms]),
    "CI_Lower": np.exp(model.conf_int().loc[segment_terms, 0]),
    "CI_Upper": np.exp(model.conf_int().loc[segment_terms, 1]),
    "P_Value": model.pvalues[segment_terms],
}).reset_index(drop=True)
adjusted_segment_results["Adjusted_P_Value"] = multipletests(adjusted_segment_results["P_Value"], method="holm")[1]
adjusted_segment_results["Significant_After_Controls"] = adjusted_segment_results["Adjusted_P_Value"] < ALPHA
display(adjusted_segment_results.style.format({"Coefficient":"{:.3f}", "Odds_Ratio":"{:.3f}", "CI_Lower":"{:.3f}", "CI_Upper":"{:.3f}", "P_Value":"{:.3g}", "Adjusted_P_Value":"{:.3g}"}))
print(f"McFadden pseudo-R²: {model.prsquared:.3f}")


### What we can see and learn from this output

- This model asks a different question from the raw default-rate table: does the segment still matter after comparing clients with similar measured risk characteristics?
- Segment 3 remains significant after the Holm correction. Its adjusted odds of default are about **1.43 times** those of Segment 1, with a confidence interval of about **1.10 to 1.84**.
- Segment 2 has adjusted odds of **1.27**, but its corrected p-value is **0.074**, so it is not significant at the 5% level.
- Segment 4 has adjusted odds of **1.28**, with the same corrected p-value of about **0.074**, so it is also not significant at the 5% level.
- McFadden's pseudo-R-squared is **0.377**. This suggests useful model fit, but it is not the same as accuracy or “37.7% explained.”

**Main lesson:** the large raw differences for Segments 2 and 4 are mostly explained by their client characteristics. Segment 3 still contains additional risk information that the included variables do not fully explain.

## 11. Stability over time

In [ ]:
mix = pd.crosstab(df[YEAR_COL], df[SEGMENT_COL], normalize="index") * 100
plt.figure(figsize=(10, 5))
sns.heatmap(mix.T, annot=True, fmt=".1f", cmap="Blues")
plt.title("Segment population share by underwriting year (%)")
plt.xlabel("Underwriting year"); plt.ylabel("Segment")
plt.show()

rates_over_time = df.pivot_table(index=YEAR_COL, columns=SEGMENT_COL, values=TARGET_COL, aggfunc="mean") * 100
rates_over_time.plot(marker="o", figsize=(10, 5))
plt.ylabel("Default rate (%)"); plt.title("Segment default rate over time")
plt.legend(title="Segment"); plt.show()

year_segment_table = pd.crosstab(df[YEAR_COL], df[SEGMENT_COL])
chi2_year, p_year, _, _ = stats.chi2_contingency(year_segment_table)
v_year = np.sqrt(chi2_year / (year_segment_table.to_numpy().sum() * min(year_segment_table.shape[0]-1, year_segment_table.shape[1]-1)))
print(f"Year/segment association: p={p_year:.4g}, Cramer's V={v_year:.3f}")


### What we can see and learn from this output

- The segment shares stay broadly similar across underwriting years.
- The year/segment p-value is **0.816**, so there is no evidence that segment assignment changed systematically by year.
- Cramer's V is **0.017**, which is extremely small.
- The yearly default-rate lines may move slightly because of normal sample variation, so large future changes should still be monitored.

**Main lesson:** the segmentation mix appears stable over time in this synthetic dataset.

## 12. Automated model-risk summary

In [ ]:
large_features = numeric_tests.loc[numeric_tests["Practical_Effect"] == "Large", "Feature"].tolist()
medium_features = numeric_tests.loc[numeric_tests["Practical_Effect"] == "Medium", "Feature"].tolist()
material_pairs = pairwise_default.loc[pairwise_default["Material_Difference"], ["Segment_A", "Segment_B"]].values.tolist()
residual_terms = adjusted_segment_results.loc[adjusted_segment_results["Significant_After_Controls"], "Term"].tolist()

print("SEGMENTATION VALIDATION SUMMARY")
print("-" * 35)
print(f"Segment reliability gate passed: {not segment_quality['Segment_Adequacy_Warning'].any()}")
print(f"Features with large separation: {large_features or 'None'}")
print(f"Features with medium separation: {medium_features or 'None'}")
print(f"Segment pairs with material default-rate gaps: {material_pairs or 'None'}")
print(f"Segment effects remaining after controls: {residual_terms or 'None'}")
print(f"Time/segment association strength (Cramer's V): {v_year:.3f}")

print("\nReviewer interpretation:")
print("1. Strong feature separation supports distinct profiles, but very large separation may also indicate rule-driven or circular construction.")
print("2. Default-rate differences should be evaluated using both confidence intervals and practical gaps.")
print("3. Residual segment coefficients may indicate information not captured by controls, misspecification, or embedded policy effects; they are not automatically causal.")
print("4. Synthetic labels provide no fairness evidence. A real review requires approved protected-attribute/proxy testing, governance, and legal review.")


### What we can see and learn from this output

- All segments pass the **outcome reliability and precision check**: there are enough defaults and non-defaults, and the confidence intervals are sufficiently narrow.
- All tested features show large overall separation.
- All six pairs of segments have materially different raw default rates.
- After controlling for the measured characteristics, only Segment 3 still differs significantly from Segment 1.
- The very small time-association value (**0.017**) supports stability across underwriting years.

**Overall lesson:** the synthetic segmentation creates clear and stable client groups, and Segment 3 is the main high-risk group. Because the data was deliberately generated this way, the result validates the analysis process - not a real bank segmentation.

## 13. Export review tables

In [ ]:
output_dir = Path("../reports") if Path("../reports").exists() else Path("reports")
output_dir.mkdir(exist_ok=True)
segment_sizes.to_csv(output_dir / "segment_sizes.csv")
numeric_tests.to_csv(output_dir / "numeric_segment_tests.csv", index=False)
smd_table.to_csv(output_dir / "pairwise_smd.csv", index=False)
default_summary.to_csv(output_dir / "default_rate_summary.csv")
pairwise_default.to_csv(output_dir / "pairwise_default_tests.csv", index=False)
adjusted_segment_results.to_csv(output_dir / "adjusted_segment_model.csv", index=False)
print(f"Saved review tables to: {output_dir.resolve()}")


### What we can see and learn from this output

- The result tables were exported successfully to `/content/reports` in the Colab session.
- These CSV files provide an audit trail of the segment sizes, statistical tests, default comparisons, and adjusted model.
- Colab storage is temporary, so the files should be downloaded or copied to Google Drive before the runtime is closed.

**Main lesson:** the analysis results are ready to be saved and shared for review.

## Limitations and governance notes

- The data is synthetic and deliberately embeds segment differences. Results validate the generated scenario, not a real portfolio.
- Statistical significance is sensitive to sample size; effect sizes and confidence intervals must drive interpretation.
- Segment membership should not be treated as causal.
- Stability should also be assessed out-of-time and across economic regimes when real history becomes available.
- Fairness assessment requires approved variables, proxy methodology, governance thresholds, and appropriate legal/compliance review.
- Before production use, convert notebook logic into tested functions, freeze package versions, and add independent validation.